# Loading in our BERT model

* Load in the trained BERT model from 'bert_v1.ipynb'

In [11]:
import torch 
import pickle

from transformers import BertTokenizer, BertForSequenceClassification

In [12]:
# We do not have to retrain the model.
model_path = "./bert_goemotions_v1"

tokenizer = BertTokenizer.from_pretrained(model_path)

model = BertForSequenceClassification.from_pretrained(model_path)

model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9184.81it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [13]:
with open("./bert_goemotions_v1/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

In [14]:
# Test on a new sentence.

text = "I absolutely love this game!"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=1)

emotion = label_encoder.inverse_transform(
    [prediction.item()]
)[0]

print("Prediction:", emotion)

Prediction: love


In [15]:
def predict_emotion(text, model, tokenizer, label_encoder, max_length=128):
  inputs = tokenizer(
    text, 
    return_tensors="pt", 
    truncation=True, 
    max_length=max_length
  )
  
  with torch.no_grad():
    outputs = model(**inputs)
  
  prediction = torch.argmax(outputs.logits, dim=1)

  emotion = label_encoder.inverse_transform(
      [prediction.item()]
  )[0]

  return emotion

In [16]:
print("Prediction:", predict_emotion("I absolutely love this game!", model, tokenizer, label_encoder))

Prediction: love


In [17]:
print("Prediction:", predict_emotion("I absolutely hate this game!", model, tokenizer, label_encoder))

Prediction: anger
